# Introduction to Logistic Regression
## DS-3001: Foundations of Machine Learning

Content adapted from Terence Johnson (UVA)

### Loading Environment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os # For changing directory

# To mount your google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path_to_DS_3001_folder = '/content/drive/MyDrive/DS-3001/03_Tuning_Testing_Evaluating'
# path_to_DS_3001_folder = ''

# Update the path to your folder for the class
# Where you stored the data from the previous noteboook
os.chdir(path_to_DS_3001_folder)

## Introduction

- Linear models, especially with LASSO and Ridge, are one of our most powerful frameworks: We can manage complex feature spaces, and explain how the covariates/features determine the values of the predictions

- Linear models, however return a numeric value between $-\infty$ and $+\infty$, and we can't really restrict this further (e.g. probabilities between 0 and 1, count data with values like 0, 1, 2, ..., survival data with times greater than zero)

- But, we can extend the concept of linear models to accomodate this situation

## A Motivating Example
- We're going to work with a medical data set that has an outcome variable labeled `DEATH_EVENT`:
  - `DEATH_EVENT=0` means the patient survived
  - `DEATH_EVENT=1` means the patient died

- Let's make a quick linear model and take a look at the predictions it makes

- (We won't bother with the train-test split, because we're just playing with the linear model framework on this occasion)

- Since the linear model is predicting a binary outcome, we interpret its value as a probability

In [ ]:
# Import for a linear regression mode
from sklearn.linear_model import LinearRegression

# Loading in a data frame and looking at the data
cardiac_df = pd.read_csv('./data/heart_failure_clinical_records_dataset.csv')
cardiac_df.head()

In [ ]:
# Fit a linear regression model to this data


In [ ]:
# Print out the coefficients for later reference
pd.DataFrame({'Variable':model.feature_names_in_, 'Coefficient':model.coef_})

**Question: What's the main issue with the result above?**

**Answer:**

### Linear Probability Models
- **Values below 0 and above 1:**
  - In the example abov, we can see that there are a substantial number of outcomes below zero and above one. We can't interpret these as probabilities.

- **Is it okay if there's only a few violations?**
  - If there are only a few violations not far below zero or above one, we might shrug and put up with it. The linear model is very explainable, easy to compute, and so on.

- **Example where it's not okay:**
  - But if we need this for clinical purposes, this many violations of these sizes suggests the linear model isn't a good tool.

- **Do we have another alternative?**

- We're going to resolve this issue by looking at **Logistic Regression**, which is also our path to neural networks

### Linearity and Non-Linearity
- **What does a "linear model" really mean?**

- If we take two linear models, multiple them by a scalar (`s` and `r`), and add, we get a linear model back:

$$
s \times m(x,b) + r \times m(x,b') = m(x, sb + r b')
$$

- We can write this on the board to identify how this happens.

- If a function has this property, it must be a **linear model** of the form $m(x,b) = x \cdot b$

- But this kind of model doesn't place any meaningful restrictions on $\hat{y}$

### Latent Variables
- **Problem:**
  - One of our complaints about linear models for predicting a binary class or a probability is that it might take values less than zero or greater than 1

- **Solution:**
  - There are a variety of ways to fix this, but the most popular are **probit** and **logit**

- **The latent variable:**
  - The basic premise of these models is that, along with features $x_i$, there is an **unobserved** linear index or **latent variable**,
$$ L_i = b \cdot x_i$$

- **What we observe:**
  - Our observation is not the latent variable, but the result of placing the latent variable in an activation function (**$A(L_i)$**):

$$ y_i = A(L_i) = \begin{cases} 1, & L_i \ge 0 \\ 0, & L_i < 0 \end{cases} $$

- **Interpretation of the activation function:**
  - If the latent index is large: you'll observe a 1
  - Otherwise, you observe a zero

- **Beneficial trick:**
  - This is a great trick for situations where numeric covariates/variables determine categorical/binary/integer-valued outcomes

- **Non-linear function:**
  - The use of an activation function makes this model non-linear.

- **Leads to neural networks:**
  - Later on, this will be our first model of a neuron and artificial neural network (ANN)

In [ ]:
# Plot this example activation function using a grid of Latent Variabels


### Genearal Linear Models for Classification

So we take our latent index over a carefully engineered feature space. Given the outcome, we now have three types of regression:

1. **Logistic Regression**:
    - If the outcome is **binary** 0/1, we use **logistic regression** to predict a probability between 0 and 1 that the outcome is 1, $\hat{p}$

2. **Multinomial logistic regression**:
    - If the outcome is **categorical**, we use **multinomial logistic regression** or **softmax regression** to predict the probabilities of each label occurring, $(\hat{p}_1, \hat{p}_2, ... \hat{p}_L)$

3. **Poisson regression**:
   - If the outcome is **a count** like $0, 1, 2, ...$ we use **Poisson regression** to predict the probability that each non-negative integer value occurs, $(\hat{p}_0, \hat{p}_1, \hat{p}_2, ...)$

All of these methods can be used for classification instead of regression, and you report the most likely outcome as your classification

### The Activation Function Leads to non-linearity

- So now, our models are linear in the latent index because $s L + (1-r)L' = x \cdot (s b + rb')$, but they are not linear overall because of the activation function:
$$
s A(L) + r A(L') \neq A(x \cdot (s b + rb'))
$$

- This is one of the key observations that makes neural networks useful

## Logistic Regression (Binary Outcomes)

### Examples where Logistic Regression could be Benficial:

- What is the probability a test pilot loses consciousness ($y_i =1, 0$) in a high risk scenario, based on physiological data $x_i$?

- Who buys a new car or washing machine this year ($y_i =1, 0$), based on socio-economic characteristics $x_i$?

- Are you hired for a job or denied a loan ($y_i = 0, 1$), based on your qualifications $x_i$?

## Logistic Regression

- We need "nice" function that takes on the "bell-curve" shape, but is easier to compute and work with than the normal distribution.

- The **Logistic Distribution** serves this purpose and is defined as
$$
F(L) = \dfrac{e^{L}}{1+e^{L}},
$$
and has a density function
$$
f(L) = \dfrac{e^{L}}{(1+e^{L})^2} = F(L)(1-F(L))
$$

- This has the classic S-shaped distribution (CDF), and bell-shaped density (PDF)

**Exercise:** Compare the CDF for the logistic distribution and the normal (gaussian) distribution

In [ ]:
# Answer Here

# Labels for the graph
plt.xlabel("$L = b \\cdot x$")
plt.ylabel("$F(L)$")
plt.legend(loc='lower right')
plt.title('Logit and Normal CDFs')
plt.show()

**Exercise:** Plot the PDF for the logistic distribution vs the normal (gaussian) distribtuion using our latent variable

In [ ]:
# Answer here

# Labels for us to use
plt.xlabel(r"$L = b \cdot x$")
plt.ylabel(r"$f(L)$")
plt.title("Logit and Normal PDFs")
plt.legend(loc='upper right')
plt.show()

**Exercise:** Compare the outputs of linear regression and logistic regression when using the logistic distribution on our latent variable

In [ ]:
# Answer here


## Prediction with Logistic Regression

- **Output from linear regression is latent variable:**
  - Linear regression gives an output for an observation ($x_i$) using the coefficient vector (b): $x_i \cdot b$.
  - This is our *latent variable*, so we can rename it as $L_i = x_i \cdot b$

- **Turning our latent variable into a probability:**
  - With the logistic distribution, we can plug $L_i$ into $F(L)$, and we are guaranteed to get a probability back:

\begin{gather}
\hat{p}_i = \frac{e^{x_i \cdot b}}{1+e^{x_i \cdot b}} \\
= \frac{e^{L_i}}{1 + e^{L_i}}
\end{gather}

- **Interpretation:**
  - This is the probability that we observe $y_i = 1$, after observing $x_i$.

- **Why is this still regression?:**
  - We are using $x_i$ to provide an estimate of the probability, a number, so this is regression

**Exercise:** Apply the logistic distribution to the output that we had from linear regression at the beginning of the notebook to demonstrate how we go from the latent variable to probabilities.

**Note**: The linear regression model from before was trained by minimizing MSE and without the transformation we're applying now. For that reason, these predictions will not be optimal. This is just to demonstrate what we're doing in principle for this step.

In [ ]:
# Answer here


## Prediction with Logistic Classification

* The process above gives us probabilities given our observation. **But how do we get classifications from these probabilities?**

* **Applying a threshold:** We can set a classification threshold for the proability, such that probabiliites greater than that threshold are classified as positive cases and below are negative cases.

* Our predicted classification for the ith observation ($\hat{\ell}_i$) is calculated as:

$$
\hat{\ell}_i = \begin{cases}
0, & F(L) = \frac{e^{x_i \cdot b}}{1+e^{x_i \cdot b}} < .5 \\
1, & F(L) = \frac{e^{x_i \cdot b}}{1+e^{x_i \cdot b}} \ge .5.
\end{cases}
$$

* **Hard Classification:**
  - Here, the output of our model is a **hard classification** as class 0 or class 1, rather than a number between 0 and 1

* **The threshold does not need to be 0.5**:
  - There are cases where it is favorable to set the threshold for hard classification to a different value to minimize some cost based on our prediction error. This is a way to improve the performance of the model depending on the task.

**Exercise:** Apply the hard classification to the probabilites from the model above. What happens to your classifications as you change the threshold?

In [ ]:
# Answer Here


### The Logistic Model as a Neural Network

- Imagine our latent variable, $L_i = x_i \cdot b$, as a stimulus hitting a neuron.

- If the stimulus is very negative, the neuron is unlikely to fire, because $F(L)$ is close to zero:
$$
F(L) = \dfrac{e^{L}}{1+e^{L}}
$$

- If the stimulus is very positive, the neuron is likely to fire, because $F(L)$ is close to 1.
$$
\dfrac{e^{L}}{1+e^{L}}
$$

- If we start wiring together units of the form $A(b + w \cdot z)$, we get artifical neural networks. A single layer neural network is logistic regression.

  * **A():** Some activation function. Can be sigmoid, ReLU, Leaky ReLU, etc.
  * **b**: Some bias/intercept term. Similar to what we've had for linear regression.
  * **w**: Some weight that we learn.
  * **z**: Some intermediate input that came from the previous layer of the network.

![](https://raw.githubusercontent.com/ds4e/undergrad_ml/da858dfa022d60cc7fd1c9df7242a0ada3267467/src/neuron.png)

## How to Fit A Logistic Regression Model

- **Should we use SSE/MSE/RMSE?**
  - You could compute the SSE of the predicted probabilities from the observed outcomes. This is called the **Brier Score**, but most people do not use this method.

- **Using the Likelihood function:**
  - Instead, we ask, "What is the probability of observing these data, given $b$?" That is the **likelihood function**.

- **Maximum likelihood estimation**:
  - We then pick the $b$ (coefficient vector) that maximizes the likelihood function.

- Then the probability of observing these $y$ given $x$ and $b$ is given by the **likelihood function**,

\begin{gather}
  pr[y|x,b] = \prod_{i =1}^N \left(\dfrac{\exp(b \cdot x_i)}{1+\exp(b \cdot x_i)}\right)^{y_i} \times  \left(  \dfrac{1}{1+\exp(b \cdot x_i)}\right)^{1-y_i} \\
  = \prod_{i =1}^N {\hat{p}_i(b)}^{y_i} \times (1 - \hat{p}_i(b))^{1-y_i}
\end{gather}
$$ $$

- This is called **maximum likelihood estimation**, and is an alternative way to approach the problem of picking parameters $b$ to fit a model to data that has excellent statistical properties

## Our Loss Function: Binary Cross Entropy
- Because the likelihood function is a product of probabilities, we typically take the natural logarithm to convert the product into a sum. Since the log function is increasing, the two problems have the same solution.

- The **log likelihood**, which statistics people want to maximize, is
$$
\ell \ell(b) = \sum_{i=1}^N y_i \log \left( \hat{p}_i(b) \right) + (1-y_i) \log \left( 1-\hat{p}_i(b) \right)
$$

- The **binary cross entropy**, which CS people want to minimize, is
$$
\text{bce}(b) = - \sum_{i=1}^N y_i \log \left( \hat{p}_i(b) \right) + (1-y_i) \log \left( 1-\hat{p}_i(b) \right)
$$

- These are the same, it's just which community is writing the model down

- This is not a linear model

## Using Logistic Regression in SciKIt

- The SciKIt learn function for Logistic Regression is imported as such:
    - `from sklearn.linear_model import LogisticRegression`

- **Documentation:** The logistic regression is found here: [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

- **Set `penalty=None`:** It happens that **SciKit logistic regression is not logistic regression**. **You must add an option,`penalty=None`, to get vanilla logistic regression.** Otherwise, you will get the regularized Ridge/L2 version of logistic regression.

- Logistic regression also has a `fit_intercept` option to handle the dummy variable trap, just like linear regression.

- **Min-max scale your variables:** When using SciKit for nonlinear models, **min-max scale your numeric variables**. Otherwise they can be unstable.

- For a fitted model, you can use it for regression or classification:
    - `model.predict(X)` will return a class label, 0 or 1
    - `model.predict_probab(X)` will return probabilities between 0 and 1

**Exercise:** Fit a logistic regression model to the heart failure data from the beginning of the notebook

In [ ]:
# Answer here


In [ ]:
# Look at your predictions from the logistic regression model, what shape are they returned as?
# Plot the distribtuion of predicted probabilities


In [ ]:
# Gather the hard classifications given your fit model
# What is returned?


In [ ]:
# Compare the predicted probabilities with the default linear regression we used before


### How do we find the solution for Logistic Regression?

* With linear regression, we had a unique solution that we could solve by hand and minimized the MSE: $$\hat{b} = (X^T X)^{-1}X^T y$$

* We cannot do the same for for logistic regresion.

* Instead, we use **gradient descent:**
  * Starting from an initial guess of $b$, we compute the gradient of the binary cross entropy (BCE) loss function.
  * Given this gradient, we take steps in the direction in which the value of the BCE function.
  * You can take an entire class on this concept.